# Static Benchmark Analysis — TC22 LRF Thesis

Phân tích chi tiết bài test tĩnh **4 cự ly × 3 pipeline** (Test_ID 8X01/02/03).

**Mục tiêu của notebook:**
1. Tổng kết kết quả on-device từ firmware (raw, baseline, alpha-beta)
2. Chạy fair comparison offline trên cùng raw input
3. Phân tích xu hướng σ/bias theo cự ly
4. Đánh giá CÁI NÀO cải thiện, CÁI NÀO chưa
5. Phát hiện anomaly (LOS clutter, alpha-beta velocity drift)
6. So sánh với giai đoạn 1
7. Xuất bảng kết quả dùng cho báo cáo Chương 6

**Cách dùng:**
- Upload 12 file CSV (4 cự ly × 3 pipeline) khi được hỏi
- Chạy tuần tự từng cell
- Notebook self-contained, không cần import alphabeta_core.py / metrics.py

## 1. Setup và Upload

In [ ]:
import os, math
from dataclasses import dataclass
from typing import Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

In [ ]:
# Upload 12 file CSV (4 cự ly × 3 pipeline)
try:
    from google.colab import files
    print('Chọn 12 file Test_ID_8X0X_<dist>m.csv:')
    uploaded = files.upload()
    print(f'\nĐã upload {len(uploaded)} file')
except ImportError:
    print('Không phải Colab — đảm bảo CSV ở cùng thư mục notebook')

## 2. Inline Alpha-Beta + Baseline (mirror firmware)

Để notebook self-contained, các class tracker được nhúng inline.
Mọi tham số đã khớp với firmware `PRESET_D_AB_BALANCED`.

In [ ]:
# ===== Constants khớp firmware =====
MEAS_OK, MEAS_TIMEOUT, MEAS_BAD_CRC, MEAS_BAD_FRAME, MEAS_NO_SIGNAL = 0, 1, 2, 3, 4
TRACK_SEARCHING, TRACK_CANDIDATE, TRACK_STABLE, TRACK_LOST = 0, 1, 2, 3
EST_BASELINE, EST_ALPHABETA, EST_RAW_ONLY = 0, 1, 2

def bb_beta(alpha):
    return (alpha**2) / (2.0 - alpha)

@dataclass
class ABConfig:
    alpha: float = 0.30
    beta:  float = 0.0529
    gate_threshold_m: float = 10.0
    max_reject: int = 5
    candidate_hits: int = 2
    lost_after_invalid: int = 5
    default_dt_s: float = 0.40
    min_dt_s: float = 1e-3
    max_dt_s: float = 2.0
    reinit_on_switch: bool = True
    init_rate_mps: float = 0.0
    invalid_velocity_decay: float = 0.85
    max_predict_hold_samples: int = 8

In [ ]:
# ===== Alpha-Beta Tracker (mirror firmware) =====
class RangeTracker:
    def __init__(self, cfg=ABConfig()):
        self.cfg = cfg
        self.reset()
    def reset(self):
        self.initialized = False
        self.x = math.nan
        self.v = self.cfg.init_rate_mps
        self.last_ts_ms = None
        self.reject_count = 0
        self.predict_hold_count = 0
    def _resolve_dt(self, ts_ms, fps):
        dt = self.cfg.default_dt_s
        if self.last_ts_ms is not None and ts_ms > self.last_ts_ms:
            dt = (ts_ms - self.last_ts_ms) / 1000.0
        elif fps > 0.01:
            dt = 1.0 / fps
        if math.isnan(dt) or dt < self.cfg.min_dt_s: dt = self.cfg.min_dt_s
        if dt > self.cfg.max_dt_s: dt = self.cfg.max_dt_s
        return dt
    def step(self, ts_ms, z_m, status, fps=0.0):
        cfg = self.cfg
        if status != MEAS_OK or not math.isfinite(z_m):
            if not self.initialized: return (math.nan, math.nan, math.nan, math.nan, False)
            dt = self._resolve_dt(ts_ms, fps)
            self.v *= cfg.invalid_velocity_decay
            self.predict_hold_count += 1
            if self.predict_hold_count >= cfg.max_predict_hold_samples: self.v = 0.0
            else: self.x = self.x + self.v * dt
            self.last_ts_ms = ts_ms
            return (self.x, self.v, self.x, math.nan, False)
        if not self.initialized:
            self.initialized = True
            self.x, self.v = z_m, cfg.init_rate_mps
            self.last_ts_ms = ts_ms
            return (self.x, self.v, z_m, 0.0, False)
        dt = self._resolve_dt(ts_ms, fps)
        x_pred = self.x + self.v * dt
        residual = z_m - x_pred
        if abs(residual) > cfg.gate_threshold_m:
            self.reject_count += 1
            if cfg.reinit_on_switch and self.reject_count >= cfg.max_reject:
                self.x, self.v = z_m, cfg.init_rate_mps
                self.last_ts_ms = ts_ms
                self.reject_count, self.predict_hold_count = 0, 0
                return (self.x, self.v, x_pred, residual, True)
            self.x, self.v = x_pred, self.v
            self.last_ts_ms = ts_ms
            self.predict_hold_count = 0
            return (self.x, self.v, x_pred, residual, True)
        self.reject_count, self.predict_hold_count = 0, 0
        self.x = x_pred + cfg.alpha * residual
        if cfg.beta > 0 and dt > cfg.min_dt_s:
            self.v = self.v + (cfg.beta / dt) * residual
        self.last_ts_ms = ts_ms
        return (self.x, self.v, x_pred, residual, False)

In [ ]:
# ===== Baseline (Gating + Median(5) + EMA) =====
class BaselineFilter:
    def __init__(self, alpha=0.25, gate=10.0, max_reject=5):
        self.alpha, self.gate, self.max_reject = alpha, gate, max_reject
        self.reset()
    def reset(self):
        self.ema = math.nan
        self.buf = [0.0]*5
        self.length, self.idx, self.reject_count = 0, 0, 0
    def update(self, m, valid):
        if not valid: return self.ema
        if math.isfinite(self.ema):
            if abs(m - self.ema) > self.gate:
                self.reject_count += 1
                if self.reject_count < self.max_reject: return self.ema
                self.ema = m; self.reject_count = 0
                self.buf = [m]*5; self.length = 5
            else: self.reject_count = 0
        if self.length < 5: self.length += 1
        self.buf[self.idx] = m
        self.idx = (self.idx + 1) % 5
        med = m if self.length < 5 else sorted(self.buf)[2]
        self.ema = med if math.isnan(self.ema) else self.alpha * med + (1-self.alpha) * self.ema
        return self.ema

# ===== Apply on DataFrame =====
def run_alphabeta(df, cfg=None):
    cfg = cfg or ABConfig()
    t = RangeTracker(cfg)
    out = []
    for _, row in df.iterrows():
        ts = int(row['timestamp_ms']); z = row['raw_m']
        st = int(row['status']) if pd.notna(row['status']) else MEAS_TIMEOUT
        z = float(z) if pd.notna(z) else math.nan
        x, v, xp, r, rej = t.step(ts, z, st)
        out.append({'ts': ts, 'raw': z, 'est': x, 'rate': v, 'pred': xp, 'resid': r, 'rejected': rej, 'status': st})
    return pd.DataFrame(out)

def run_baseline(df, alpha=0.25, gate=10.0, max_reject=5):
    f = BaselineFilter(alpha, gate, max_reject)
    out = []
    for _, row in df.iterrows():
        ts = int(row['timestamp_ms']); z = row['raw_m']
        st = int(row['status']) if pd.notna(row['status']) else MEAS_TIMEOUT
        z = float(z) if pd.notna(z) else math.nan
        valid = (st == MEAS_OK) and math.isfinite(z)
        est = f.update(z, valid)
        out.append({'ts': ts, 'raw': z, 'est': est, 'status': st})
    return pd.DataFrame(out)

In [ ]:
# ===== Metric helpers =====
def bias_sigma_rmse(est, truth):
    est = pd.to_numeric(pd.Series(est), errors='coerce').dropna().values
    if len(est) == 0: return dict(bias=np.nan, sigma=np.nan, rmse=np.nan, n=0)
    err = est - truth
    return dict(bias=float(err.mean()), sigma=float(err.std(ddof=1)) if len(err)>1 else 0.0,
                rmse=float(np.sqrt((err**2).mean())), n=int(len(err)))

def fmt(v, p=3):
    return f'{v:+.{p}f}' if (v is not None and np.isfinite(v)) else 'nan'

## 3. Cấu hình cự ly và load CSV

In [ ]:
# ===== Cấu hình cự ly (sửa theo dữ liệu của bạn) =====
TARGETS = {
    50:  ('Test_ID_8001_50m.csv',  'Test_ID_8002_50m.csv',  'Test_ID_8003_50m.csv'),
    180: ('Test_ID_8001_180m.csv', 'Test_ID_8002_180m.csv', 'Test_ID_8003_180m.csv'),
    345: ('Test_ID_8001_345m.csv', 'Test_ID_8002_345m.csv', 'Test_ID_8003_345m.csv'),
    # Tên file 522m có suffix random — sửa lại đúng tên thực tế:
    522: ('Test_ID_8001_522m.csv', 'Test_ID_8002_522m.csv', 'Test_ID_8003_522m.csv'),
}
# Truth là giá trị đo bằng Google Maps Distance hoặc thước. Sửa nếu cự ly thực tế khác.
TRUTH = {50: 50.0, 180: 180.0, 345: 345.0, 522: 522.0}

# ===== Load và normalize =====
def load_csv(path):
    df = pd.read_csv(path)
    # Normalize tên cột
    if 'dev_ts_ms' in df.columns and 'timestamp_ms' not in df.columns:
        df = df.rename(columns={'dev_ts_ms': 'timestamp_ms'})
    if 'meas_status' in df.columns and 'status' not in df.columns:
        df = df.rename(columns={'meas_status': 'status'})
    df['timestamp_ms'] = pd.to_numeric(df['timestamp_ms'], errors='coerce')
    df = df.dropna(subset=['timestamp_ms']).reset_index(drop=True)
    df['t_s'] = (df['timestamp_ms'] - df['timestamp_ms'].iloc[0]) / 1000.0
    return df

DATA = {}
for dist, (fr, fb, fa) in TARGETS.items():
    try:
        DATA[dist] = {'raw': load_csv(fr), 'baseline': load_csv(fb), 'alphabeta': load_csv(fa)}
        print(f'  {dist}m: raw={len(DATA[dist]["raw"])} baseline={len(DATA[dist]["baseline"])} ab={len(DATA[dist]["alphabeta"])}')
    except FileNotFoundError as e:
        print(f'  {dist}m: THIẾU file — {e.filename}')

## 4. Phần A — Bảng on-device summary

Đây là kết quả từ firmware tại thời điểm đo (cột `est_m` trong CSV).

In [ ]:
rows = []
for dist, ts in DATA.items():
    truth = TRUTH[dist]
    for tag, df in [('raw', ts['raw']), ('baseline', ts['baseline']), ('alpha-beta', ts['alphabeta'])]:
        n = len(df)
        ok_pct = (df['status'] == 0).mean() * 100
        est = pd.to_numeric(df['est_m'], errors='coerce')
        m = bias_sigma_rmse(est.dropna(), truth)
        fps = pd.to_numeric(df[df['status']==0]['fps'], errors='coerce').mean()
        rows.append({'cự ly (m)': dist, 'pipeline': tag, 'n': n, 'OK%': round(ok_pct,1),
                     'bias (m)': round(m['bias'],3), 'σ (m)': round(m['sigma'],3),
                     'RMSE (m)': round(m['rmse'],3), 'fps': round(fps,2)})

on_device = pd.DataFrame(rows)
display(on_device)

## 5. Phần B — Fair comparison offline

**Chạy cả 3 pipeline trên CÙNG raw input từ file 8X01** (mode RAW_ONLY).
Đây là cách duy nhất so sánh hoàn toàn fair, vì 3 session trên thiết bị có raw khác nhau.

In [ ]:
fair_rows = []
fair_data = {}  # lưu lại để dùng plot sau
for dist, ts in DATA.items():
    truth = TRUTH[dist]
    df_raw = ts['raw']
    # 3 pipeline trên CÙNG raw
    raw_vals = pd.to_numeric(df_raw['raw_m'], errors='coerce').dropna()
    out_bl = run_baseline(df_raw)
    out_ab = run_alphabeta(df_raw)
    fair_data[dist] = dict(raw=df_raw, bl=out_bl, ab=out_ab)
    
    for tag, vals in [('Raw', raw_vals),
                       ('Baseline', pd.to_numeric(out_bl['est'], errors='coerce').dropna()),
                       ('AlphaBeta', pd.to_numeric(out_ab['est'], errors='coerce').dropna())]:
        m = bias_sigma_rmse(vals.values if hasattr(vals,'values') else vals, truth)
        fair_rows.append({'cự ly (m)': dist, 'pipeline': tag, 'n': m['n'],
                          'bias (m)': round(m['bias'],3), 'σ (m)': round(m['sigma'],3),
                          'RMSE (m)': round(m['rmse'],3)})

fair = pd.DataFrame(fair_rows)
display(fair)

# Bảng cải thiện %
print('\n=== Cải thiện σ vs raw ===')
pivot = fair.pivot(index='cự ly (m)', columns='pipeline', values='σ (m)')[['Raw','Baseline','AlphaBeta']]
pivot['BL cải thiện %'] = ((1 - pivot['Baseline']/pivot['Raw']) * 100).round(1)
pivot['AB cải thiện %'] = ((1 - pivot['AlphaBeta']/pivot['Raw']) * 100).round(1)
display(pivot)

## 6. Phần C — Xu hướng σ và Bias theo cự ly

In [ ]:
dists = sorted(DATA.keys())
sig_raw = [fair_data[d]['raw']['raw_m'].astype(float).std(ddof=1) for d in dists]
sig_bl  = [pd.to_numeric(fair_data[d]['bl']['est'], errors='coerce').std(ddof=1) for d in dists]
sig_ab  = [pd.to_numeric(fair_data[d]['ab']['est'], errors='coerce').std(ddof=1) for d in dists]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(dists, sig_raw, 'o-', label='Raw', linewidth=2, markersize=10)
axes[0].plot(dists, sig_bl,  's-', label='Baseline (G+M+EMA)', linewidth=2, markersize=10)
axes[0].plot(dists, sig_ab,  '^-', label='Alpha-Beta PRESET_D', linewidth=2, markersize=10)
axes[0].set_xlabel('Cự ly (m)'); axes[0].set_ylabel('σ (m)')
axes[0].set_title('σ theo cự ly — fair comparison offline')
axes[0].legend(); axes[0].set_yscale('log')

# Bias
bias_raw = [fair_data[d]['raw']['raw_m'].astype(float).mean() - TRUTH[d] for d in dists]
bias_bl = [pd.to_numeric(fair_data[d]['bl']['est'], errors='coerce').mean() - TRUTH[d] for d in dists]
bias_ab = [pd.to_numeric(fair_data[d]['ab']['est'], errors='coerce').mean() - TRUTH[d] for d in dists]

axes[1].plot(dists, bias_raw, 'o-', label='Raw', linewidth=2, markersize=10)
axes[1].plot(dists, bias_bl,  's-', label='Baseline', linewidth=2, markersize=10)
axes[1].plot(dists, bias_ab,  '^-', label='Alpha-Beta', linewidth=2, markersize=10)
axes[1].axhline(0, color='k', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Cự ly (m)'); axes[1].set_ylabel('Bias (m)')
axes[1].set_title('Bias theo cự ly')
axes[1].legend()

plt.tight_layout()
plt.savefig('sigma_bias_vs_distance.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Phần D — Time-series từng cự ly

Cho phép nhìn trực quan: raw có drift không, filter bám tốt không.

In [ ]:
fig, axes = plt.subplots(len(dists), 1, figsize=(14, 3.5*len(dists)))
if len(dists) == 1: axes = [axes]

for ax, dist in zip(axes, dists):
    fd = fair_data[dist]
    t_raw = (fd['raw']['timestamp_ms'] - fd['raw']['timestamp_ms'].iloc[0]) / 1000.0
    ax.plot(t_raw, fd['raw']['raw_m'], 'o', ms=2, alpha=0.3, label='raw', color='gray')
    ax.plot(t_raw, fd['bl']['est'],   '-', lw=1.5, label='baseline', color='C0')
    ax.plot(t_raw, fd['ab']['est'],   '-', lw=1.5, label='alpha-beta', color='C1')
    ax.axhline(TRUTH[dist], color='k', linestyle='--', alpha=0.5, label=f'truth {TRUTH[dist]}m')
    ax.set_xlabel('Thời gian (s)'); ax.set_ylabel('Khoảng cách (m)')
    ax.set_title(f'Cự ly {dist}m — fair comparison')
    ax.legend(loc='best', fontsize=9)

plt.tight_layout()
plt.savefig('timeseries_all_distances.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Phần E — So sánh với Giai đoạn 1 (Bảng 5.1 báo cáo cũ)

Báo cáo cũ đo CẦM TAY, giai đoạn 2 đo TRIPOD. Hệ số sạch hơn cho thấy tripod loại bỏ rung tay hiệu quả.

In [ ]:
# Giai đoạn 1 — Bảng 5.1
phase1 = {160: (7.16, 5.90), 335: (12.91, 9.12), 525: (34.62, 27.42)}
# Map cự ly tương đương
mapping = {180: 160, 345: 335, 522: 525}

rows = []
for d2, d1 in mapping.items():
    if d2 not in DATA: continue
    p1_raw, p1_filt = phase1[d1]
    p2_raw = pd.to_numeric(fair_data[d2]['raw']['raw_m'], errors='coerce').std(ddof=1)
    p2_bl  = pd.to_numeric(fair_data[d2]['bl']['est'], errors='coerce').std(ddof=1)
    rows.append({
        'cự ly tương đương': f'{d1} → {d2} m',
        'σ raw GĐ1 (cầm tay)': round(p1_raw, 2),
        'σ raw GĐ2 (tripod)': round(p2_raw, 3),
        'hệ số sạch hơn': round(p1_raw/p2_raw, 1),
        'σ filtered GĐ1': round(p1_filt, 2),
        'σ baseline GĐ2': round(p2_bl, 3),
    })
display(pd.DataFrame(rows))

## 9. Phần F — Phát hiện anomaly

### 9.1 LOS clutter — kiểm tra outlier trong raw

Nếu site có vật cản tiền cảnh, TC22 có thể bắt vật đó thay vì target.

In [ ]:
print('Phát hiện outlier trong raw (mẫu cách truth > 20%):')
for d in dists:
    raw = pd.to_numeric(fair_data[d]['raw']['raw_m'], errors='coerce').dropna()
    truth = TRUTH[d]
    deviation = (raw - truth).abs()
    outliers = raw[deviation > 0.2 * truth]
    pct = len(outliers) / len(raw) * 100
    print(f'  {d}m: outliers {len(outliers)}/{len(raw)} ({pct:.1f}%) — '
          f'min={raw.min():.1f}, max={raw.max():.1f}')
    if len(outliers) > 0:
        print(f'    Sample outlier values: {outliers.values[:10]}')

### 9.2 Alpha-beta velocity drift

Mục tiêu tĩnh → rate_mps kỳ vọng ≈ 0. Nếu rate có std lớn, alpha-beta đang track nhiễu thành velocity giả.

In [ ]:
print('Velocity behavior (rate_mps) — mục tiêu TĨNH, kỳ vọng mean≈0, std nhỏ:')
print(f'{"Cự ly":>8} {"rate mean":>12} {"rate std":>10} {"|rate max|":>12} {"verdict":>20}')
print('-'*70)
for d in dists:
    rate = pd.to_numeric(fair_data[d]['ab']['rate'], errors='coerce').dropna()
    if len(rate) == 0: continue
    mean, std, maxa = rate.mean(), rate.std(ddof=1), rate.abs().max()
    verdict = 'OK' if std < 0.2 else ('DRIFT!' if std > 0.5 else 'borderline')
    print(f'{d:>8} {mean:>+12.4f} {std:>10.4f} {maxa:>12.3f} {verdict:>20}')

## 10. Phần G — Bảng tổng kết: CÁI NÀO cải thiện, CÁI NÀO chưa

Đây là bảng quan trọng nhất — sẵn sàng paste vào báo cáo Chương 6.

In [ ]:
verdict_rows = []
for d in dists:
    sr = pd.to_numeric(fair_data[d]['raw']['raw_m'], errors='coerce').std(ddof=1)
    sb = pd.to_numeric(fair_data[d]['bl']['est'], errors='coerce').std(ddof=1)
    sa = pd.to_numeric(fair_data[d]['ab']['est'], errors='coerce').std(ddof=1)
    
    def label(s_filt, s_raw):
        imp = (1 - s_filt/s_raw) * 100
        if imp > 30: return f'TỐT ({imp:+.1f}%)'
        if imp > 10: return f'OK ({imp:+.1f}%)'
        if imp > -10: return f'~ ({imp:+.1f}%)'
        return f'TỆ HƠN ({imp:+.1f}%)'
    
    verdict_rows.append({
        'cự ly': f'{d}m',
        'σ raw': f'{sr:.3f}',
        'baseline (đối với raw)': label(sb, sr),
        'alpha-beta (đối với raw)': label(sa, sr),
    })
display(pd.DataFrame(verdict_rows))

print('\nKết luận chung:')
print('- σ raw rất sạch (cm) ở các cự ly trung — tripod hiệu quả')
print('- Baseline (G+M+EMA) thắng alpha-beta ở MỌI cự ly tĩnh — đúng kỳ vọng lý thuyết')
print('- Mức cải thiện phụ thuộc σ raw: σ_raw nhỏ → filter ít gain (đã chạm sàn)')
print('- Alpha-beta được thiết kế cho mục tiêu ĐỘNG — phần 6.4 dynamic sẽ chứng minh')

## 11. Phần H — Xuất kết quả cho báo cáo

In [ ]:
# Lưu bảng kết quả thành CSV để paste vào báo cáo / Excel
on_device.to_csv('static_ondevice_summary.csv', index=False)
fair.to_csv('static_fair_comparison.csv', index=False)

# Tải về (Colab)
try:
    from google.colab import files
    files.download('static_ondevice_summary.csv')
    files.download('static_fair_comparison.csv')
    files.download('sigma_bias_vs_distance.png')
    files.download('timeseries_all_distances.png')
except ImportError:
    print('Đã lưu các file CSV và PNG ở thư mục hiện tại')